# Wikidata Citation Enrichment via Semantic Scholar

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

---

## What This Notebook Does

This notebook enriches Wikidata scholarly articles with citation relationships (P2860 "cites work") using the Semantic Scholar Academic Graph API. Given a list of DOIs, it queries Semantic Scholar for each paper's references (what it cites), looks up both the source and cited works in Wikidata, and generates QuickStatements to add the citation links.

Citations are a crucial part of scholarly metadata that connect articles into a web of knowledge. By adding P2860 statements, articles become queryable by what they cite, enabling bibliometric analysis, citation network visualization, and research discovery through Wikidata's SPARQL endpoints.

## Key Features

- **Dual Input Methods**: Enter DOIs manually or upload CrossRef-style CSV files
- **Semantic Scholar Integration**: Leverages the free Academic Graph API for citation data
- **DOI-Based Matching**: Only adds citations where both source and cited works exist in Wikidata
- **Batch Processing**: Handles multiple articles with rate limiting and progress tracking
- **Reference Attribution**: Documents Semantic Scholar as stated source for provenance
- **Scholarly Endpoint Aware**: Uses `query-scholarly.wikidata.org` (required since May 2025)

## Workflow

1. **Input DOIs**: Paste DOIs directly or upload a CrossRef CSV export
2. **Wikidata Lookup**: Find the QID for each source article via the scholarly endpoint
3. **Semantic Scholar Query**: Fetch references for each paper from the Academic Graph API
4. **Citation Resolution**: For each reference with a DOI, look up its Wikidata QID
5. **Generate QuickStatements**: Create P2860 statements linking citing to cited works
6. **Export**: Download QuickStatements file for batch upload

## Semantic Scholar API

This notebook uses the free tier of the [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/). The free tier allows:
- 100 requests per 5 minutes
- Paper lookups by DOI, arXiv ID, MAG ID, etc.
- Reference and citation lists with metadata

For heavy usage, consider requesting an API key from Semantic Scholar.

## Output Format

```
SOURCE_QID|P2860|CITED_QID|S248|Q22908627|S854|"semantic_scholar_url"
```

Where:
- `P2860` = cites work property
- `S248|Q22908627` = stated in: Semantic Scholar
- `S854` = reference URL to Semantic Scholar paper page

## Important Notes

- Only adds citations where **both** the citing article AND cited work exist in Wikidata
- Does not create new article items—use the Article Importer notebook for that first
- Semantic Scholar coverage varies; not all DOIs have reference data
- Some references may lack DOIs, making Wikidata matching impossible

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Citation

If you use this notebook, please cite:

> Artz, Matt. (2026). MattArtzAnthro/wikidata-tools. Zenodo. https://doi.org/10.5281/zenodo.18912858

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup and Installation

*Install required Python packages and import necessary libraries.*

In [ ]:
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import re
import time
import json
from datetime import datetime
from collections import defaultdict
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO
import os

print("✓ Setup complete.")

## Configuration

*Define API endpoints, constants, and styling.*

**CRITICAL**: Since May 2025, scholarly articles are ONLY on `query-scholarly.wikidata.org`. The main Wikidata endpoint no longer contains scholarly articles!

In [ ]:
# Wikidata endpoints
# Since May 2025, scholarly articles are ONLY on the scholarly endpoint
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"
MAIN_ENDPOINT = "https://query.wikidata.org/sparql"

# Semantic Scholar API
SEMANTIC_SCHOLAR_API = "https://api.semanticscholar.org/graph/v1"
SEMANTIC_SCHOLAR_PAPER_URL = f"{SEMANTIC_SCHOLAR_API}/paper"

# Wikidata Q-IDs for references
SEMANTIC_SCHOLAR_QID = "Q22908627"  # Semantic Scholar

# Rate limiting settings
# Free tier: 100 requests per 5 minutes = ~1 request every 3 seconds
SEMANTIC_SCHOLAR_DELAY = 3.5  # seconds between requests (with buffer)
WIKIDATA_DELAY = 0.5  # seconds between Wikidata queries

# User agent for API requests
USER_AGENT = "WikidataCitationEnrichment/1.0 (Wikidata scholarly metadata project; https://www.mattartz.me/)"

# Color palette for styling
COLORS = {
    'bg_primary': '#E7ECEF',
    'text_primary': '#274C77',
    'interactive': '#6096BA',
    'bg_secondary': '#A3CEF1',
    'neutral': '#8B8C89',
    'success': '#28a745',
    'warning': '#ffc107',
    'error': '#dc3545'
}

# Container style
CONTAINER_STYLE = f"""
    background-color: {COLORS['bg_primary']};
    border-left: 5px solid {COLORS['text_primary']};
    border-radius: 10px;
    padding: 15px;
    margin: 10px 0;
"""

print(f"✓ Scholarly endpoint: {SCHOLARLY_ENDPOINT}")
print(f"✓ Semantic Scholar API: {SEMANTIC_SCHOLAR_API}")
print(f"✓ Rate limiting: {SEMANTIC_SCHOLAR_DELAY}s (S2), {WIKIDATA_DELAY}s (WD)")

## Helper Functions: DOI Utilities

*Functions for DOI parsing, cleaning, and validation.*

In [ ]:
def clean_doi(doi_input):
    """
    Extract and clean DOI from various formats.
    Returns normalized DOI or None if invalid.
    """
    if not doi_input or not isinstance(doi_input, str):
        return None

    doi_input = str(doi_input).strip()

    # Extract DOI from URLs or prefixes
    patterns = [
        r'https?://(?:dx\.)?doi\.org/(.+)',
        r'doi:(.+)',
        r'DOI:\s*(.+)'
    ]

    for pattern in patterns:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1).strip()
            break

    # Validate DOI format (must start with 10.)
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip()

    return None


def doi_to_upper(doi):
    """Convert DOI to uppercase for Wikidata matching."""
    return doi.upper() if doi else None


def parse_dois_from_text(text):
    """
    Parse DOIs from text input (one per line or comma-separated).
    Returns list of cleaned, valid DOIs.
    """
    dois = []

    # Split on newlines and commas
    lines = re.split(r'[\n,]', text)

    for line in lines:
        line = line.strip()
        if line:
            doi = clean_doi(line)
            if doi and doi not in dois:
                dois.append(doi)

    return dois


def extract_dois_from_csv(df):
    """
    Extract DOIs from a DataFrame, looking for common DOI column names.
    Returns list of cleaned, valid DOIs.
    """
    # Common DOI column names
    doi_columns = ['DOI', 'doi', 'Doi', 'DOIs', 'dois']

    doi_col = None
    for col in doi_columns:
        if col in df.columns:
            doi_col = col
            break

    if doi_col is None:
        # Try to find any column containing 'doi'
        for col in df.columns:
            if 'doi' in col.lower():
                doi_col = col
                break

    if doi_col is None:
        return []

    dois = []
    for val in df[doi_col].dropna():
        doi = clean_doi(str(val))
        if doi and doi not in dois:
            dois.append(doi)

    return dois


print("✓ DOI utility functions loaded.")

## Helper Functions: Wikidata SPARQL

*Functions to query Wikidata scholarly endpoint for article lookups.*

In [ ]:
def sparql_query(endpoint, query, timeout=60):
    """
    Execute a SPARQL query and return results.
    Returns list of result bindings or empty list on error.
    """
    try:
        response = requests.get(
            endpoint,
            params={"query": query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=timeout
        )
        response.raise_for_status()
        return response.json().get("results", {}).get("bindings", [])
    except requests.exceptions.Timeout:
        print(f"   ⚠ SPARQL timeout")
        return []
    except requests.exceptions.RequestException as e:
        print(f"   ⚠ SPARQL error: {e}")
        return []
    except Exception as e:
        print(f"   ⚠ Unexpected error: {e}")
        return []


def lookup_article_by_doi(doi):
    """
    Look up a scholarly article in Wikidata by DOI.
    Returns dict with QID and label, or None if not found.
    """
    doi_clean = clean_doi(doi)
    if not doi_clean:
        return None

    doi_upper = doi_to_upper(doi_clean)

    query = f"""
    SELECT ?article ?articleLabel WHERE {{
      ?article wdt:P356 "{doi_upper}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT 1
    """

    results = sparql_query(SCHOLARLY_ENDPOINT, query)

    if results:
        article_uri = results[0].get('article', {}).get('value', '')
        qid = article_uri.split('/')[-1] if article_uri else None
        label = results[0].get('articleLabel', {}).get('value', '')

        return {
            'qid': qid,
            'label': label,
            'doi': doi_clean
        }

    return None


def batch_lookup_dois(dois, progress_callback=None):
    """
    Look up multiple DOIs in Wikidata.
    Returns dict mapping DOI -> article info.
    """
    results = {}
    total = len(dois)

    for i, doi in enumerate(dois):
        doi_clean = clean_doi(doi)
        if doi_clean:
            result = lookup_article_by_doi(doi_clean)
            if result:
                results[doi_clean] = result
            time.sleep(WIKIDATA_DELAY)

        if progress_callback:
            progress_callback(i + 1, total)

    return results


def check_existing_citations(source_qid):
    """
    Get existing P2860 (cites work) statements for an article.
    Returns set of cited work QIDs.
    """
    query = f"""
    SELECT ?citedWork WHERE {{
      wd:{source_qid} wdt:P2860 ?citedWork .
    }}
    """

    results = sparql_query(SCHOLARLY_ENDPOINT, query)

    existing = set()
    for r in results:
        cited_uri = r.get('citedWork', {}).get('value', '')
        cited_qid = cited_uri.split('/')[-1] if cited_uri else None
        if cited_qid:
            existing.add(cited_qid)

    return existing


print("✓ Wikidata SPARQL functions loaded.")

## Helper Functions: Semantic Scholar API

*Functions to query Semantic Scholar for paper references.*

In [ ]:
def get_semantic_scholar_paper(doi, api_key=None):
    """
    Look up a paper in Semantic Scholar by DOI.
    Returns paper metadata including Semantic Scholar ID.
    """
    doi_clean = clean_doi(doi)
    if not doi_clean:
        return None

    # Semantic Scholar paper lookup by DOI
    url = f"{SEMANTIC_SCHOLAR_PAPER_URL}/DOI:{doi_clean}"
    params = {
        'fields': 'paperId,externalIds,title,url'
    }

    headers = {'User-Agent': USER_AGENT}
    if api_key:
        headers['x-api-key'] = api_key

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)

        if response.status_code == 404:
            return None  # Paper not in Semantic Scholar

        response.raise_for_status()
        data = response.json()

        return {
            'paper_id': data.get('paperId'),
            'title': data.get('title'),
            'url': data.get('url'),
            'doi': doi_clean,
            'external_ids': data.get('externalIds', {})
        }

    except requests.exceptions.RequestException as e:
        print(f"   ⚠ Semantic Scholar API error: {e}")
        return None


def get_paper_references(paper_id, api_key=None, limit=500):
    """
    Get references (papers cited BY this paper) from Semantic Scholar.
    Returns list of reference dicts with DOIs where available.
    """
    url = f"{SEMANTIC_SCHOLAR_PAPER_URL}/{paper_id}/references"
    params = {
        'fields': 'paperId,externalIds,title,url',
        'limit': limit
    }

    headers = {'User-Agent': USER_AGENT}
    if api_key:
        headers['x-api-key'] = api_key

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()

        # Handle cases where data is None or 'data' key is missing or 'data' value is null
        references = []
        for ref in (data.get('data') or []):
            cited = ref.get('citedPaper', {})
            if cited:
                external_ids = cited.get('externalIds', {})
                ref_doi = external_ids.get('DOI')

                references.append({
                    'paper_id': cited.get('paperId'),
                    'title': cited.get('title'),
                    'doi': ref_doi,
                    'url': cited.get('url'),
                    'external_ids': external_ids
                })

        return references

    except requests.exceptions.RequestException as e:
        print(f"   ⚠ Error fetching references: {e}")
        return []


def get_paper_citations(paper_id, api_key=None, limit=500):
    """
    Get citations (papers that cite this paper) from Semantic Scholar.
    Returns list of citing paper dicts with DOIs where available.

    NOTE: This is the inverse of references - use for building "cited by" relationships.
    """
    url = f"{SEMANTIC_SCHOLAR_PAPER_URL}/{paper_id}/citations"
    params = {
        'fields': 'paperId,externalIds,title,url',
        'limit': limit
    }

    headers = {'User-Agent': USER_AGENT}
    if api_key:
        headers['x-api-key'] = api_key

    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()

        # Handle cases where data is None or 'data' key is missing or 'data' value is null
        citations = []
        for cit in (data.get('data') or []):
            citing = cit.get('citingPaper', {})
            if citing:
                external_ids = citing.get('externalIds', {})
                cit_doi = external_ids.get('DOI')

                citations.append({
                    'paper_id': citing.get('paperId'),
                    'title': citing.get('title'),
                    'doi': cit_doi,
                    'url': citing.get('url'),
                    'external_ids': external_ids
                })

        return citations

    except requests.exceptions.RequestException as e:
        print(f"   ⚠ Error fetching citations: {e}")
        return []


print("✓ Semantic Scholar API functions loaded.")

## Helper Functions: QuickStatements Generation

*Functions to generate QuickStatements for adding citations.*

In [ ]:
def escape_qs_string(s):
    """Escape a string for QuickStatements."""
    if not s:
        return ''
    return str(s).replace('"', '\\"').replace('\n', ' ').strip()


def generate_citation_statement(source_qid, cited_qid, s2_url=None):
    """
    Generate a QuickStatements line for a P2860 (cites work) statement.
    Includes Semantic Scholar as stated source.

    Format: SOURCE|P2860|CITED|S248|Q22908627|S854|"url"
    """
    # Base statement
    statement = f"{source_qid}|P2860|{cited_qid}"

    # Add Semantic Scholar as source
    statement += f"|S248|{SEMANTIC_SCHOLAR_QID}"

    # Add reference URL if available
    if s2_url:
        statement += f'|S854|"{escape_qs_string(s2_url)}"'

    return statement


def generate_batch_quickstatements(citation_data):
    """
    Generate QuickStatements for all citations in the batch.

    Input: List of dicts with keys:
        - source_qid: QID of citing article
        - cited_qid: QID of cited article
        - s2_url: Semantic Scholar URL for the cited paper

    Returns: List of QuickStatements lines
    """
    statements = []

    for citation in citation_data:
        source_qid = citation.get('source_qid')
        cited_qid = citation.get('cited_qid')
        s2_url = citation.get('s2_url')

        if source_qid and cited_qid:
            stmt = generate_citation_statement(source_qid, cited_qid, s2_url)
            statements.append(stmt)

    return statements


print("✓ QuickStatements functions loaded.")

## Main Processing Function

*Core workflow for processing DOIs and generating citation statements.*

In [ ]:
def process_article_citations(doi, api_key=None, skip_existing=True, output_widget=None):
    """
    Process a single article: get its citations and resolve them to Wikidata.

    Args:
        doi: DOI of the source article
        api_key: Optional Semantic Scholar API key
        skip_existing: If True, skip citations that already exist in Wikidata
        output_widget: Optional widget for logging

    Returns:
        dict with source info, references found, and citation statements to add
    """
    def log(msg):
        if output_widget:
            with output_widget:
                print(msg)
        else:
            print(msg)

    result = {
        'doi': doi,
        'source_article': None,
        's2_paper': None,
        'total_references': 0,
        'references_with_doi': 0,
        'references_in_wikidata': 0,
        'new_citations': [],
        'skipped_existing': 0,
        'status': 'pending'
    }

    # Step 1: Look up source article in Wikidata
    log(f"  → Looking up DOI in Wikidata...")
    source_article = lookup_article_by_doi(doi)

    if not source_article:
        log(f"  ✗ Article not found in Wikidata")
        result['status'] = 'not_in_wikidata'
        return result

    result['source_article'] = source_article
    source_qid = source_article['qid']
    log(f"  ✓ Found: {source_qid} - {source_article['label'][:50]}...")

    # Check existing citations if requested
    existing_citations = set()
    if skip_existing:
        existing_citations = check_existing_citations(source_qid)
        if existing_citations:
            log(f"  ℹ Article already has {len(existing_citations)} P2860 citations")

    time.sleep(WIKIDATA_DELAY)

    # Step 2: Look up paper in Semantic Scholar
    log(f"  → Querying Semantic Scholar...")
    time.sleep(SEMANTIC_SCHOLAR_DELAY)

    s2_paper = get_semantic_scholar_paper(doi, api_key)

    if not s2_paper:
        log(f"  ✗ Paper not found in Semantic Scholar")
        result['status'] = 'not_in_s2'
        return result

    result['s2_paper'] = s2_paper
    log(f"  ✓ Found in Semantic Scholar: {s2_paper['paper_id']}")

    # Step 3: Get references from Semantic Scholar
    log(f"  → Fetching references...")
    time.sleep(SEMANTIC_SCHOLAR_DELAY)

    references = get_paper_references(s2_paper['paper_id'], api_key)
    result['total_references'] = len(references)

    if not references:
        log(f"  ℹ No references found in Semantic Scholar")
        result['status'] = 'no_references'
        return result

    # Filter to references with DOIs
    refs_with_doi = [r for r in references if r.get('doi')]
    result['references_with_doi'] = len(refs_with_doi)
    log(f"  ✓ Found {len(references)} references ({len(refs_with_doi)} with DOIs)")

    if not refs_with_doi:
        log(f"  ℹ No references have DOIs")
        result['status'] = 'no_dois'
        return result

    # Step 4: Look up each reference DOI in Wikidata
    log(f"  → Resolving references in Wikidata...")
    citations_to_add = []

    for i, ref in enumerate(refs_with_doi):
        ref_doi = ref['doi']

        # Look up in Wikidata
        cited_article = lookup_article_by_doi(ref_doi)
        time.sleep(WIKIDATA_DELAY)

        if cited_article:
            cited_qid = cited_article['qid']

            # Check if already cited
            if skip_existing and cited_qid in existing_citations:
                result['skipped_existing'] += 1
                continue

            result['references_in_wikidata'] += 1
            citations_to_add.append({
                'source_qid': source_qid,
                'source_doi': doi,
                'cited_qid': cited_qid,
                'cited_doi': ref_doi,
                'cited_title': ref.get('title', ''),
                's2_url': ref.get('url', '')
            })

    result['new_citations'] = citations_to_add
    result['status'] = 'success'

    log(f"  ✓ Found {result['references_in_wikidata']} references in Wikidata")
    if result['skipped_existing'] > 0:
        log(f"  ℹ Skipped {result['skipped_existing']} existing citations")
    log(f"  ✓ {len(citations_to_add)} new citations to add")

    return result


print("✓ Main processing function loaded.")

## Interactive Interface

*Widgets for DOI input, file upload, and processing controls.*

In [ ]:
# State storage
state = {
    'dois': [],
    'results': [],
    'all_citations': [],
    'quickstatements': []
}

# Widgets
header_html = widgets.HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📚 Citation Enrichment</h3>
    <p>Add P2860 (cites work) statements to Wikidata articles using Semantic Scholar citation data.</p>
</div>
""")

# API key input (optional)
api_key_input = widgets.Text(
    placeholder='Optional - for higher rate limits',
    description='S2 API Key:',
    layout=widgets.Layout(width='400px')
)

# Tab for input methods
doi_text = widgets.Textarea(
    placeholder='Enter DOIs, one per line:\n10.1111/example.001\n10.1234/another.002',
    layout=widgets.Layout(width='100%', height='200px')
)

upload_btn = widgets.FileUpload(
    accept='.csv,.xlsx,.xls',
    multiple=False,
    description='Upload CSV'
)

upload_status = widgets.HTML('')

# Options
skip_existing_check = widgets.Checkbox(
    value=True,
    description='Skip citations that already exist in Wikidata',
    layout=widgets.Layout(width='400px')
)

# Buttons
parse_btn = widgets.Button(
    description='Parse DOIs',
    button_style='info',
    icon='list'
)

process_btn = widgets.Button(
    description='Process Citations',
    button_style='primary',
    icon='play',
    disabled=True
)

export_btn = widgets.Button(
    description='Export QuickStatements',
    button_style='success',
    icon='download',
    disabled=True
)

# Progress
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

status_html = widgets.HTML('')
output = widgets.Output()


def on_upload(change):
    """Handle file upload."""
    if upload_btn.value:
        uploaded = list(upload_btn.value.values())[0]
        filename = uploaded['metadata']['name']
        content = uploaded['content']

        try:
            if filename.endswith('.csv'):
                df = pd.read_csv(BytesIO(content))
            else:
                df = pd.read_excel(BytesIO(content))

            dois = extract_dois_from_csv(df)

            if dois:
                state['dois'] = dois
                upload_status.value = f'<span style="color: {COLORS["success"]};">✓ Found {len(dois)} DOIs in {filename}</span>'
                process_btn.disabled = False
            else:
                upload_status.value = f'<span style="color: {COLORS["error"]};">✗ No DOIs found in {filename}</span>'
        except Exception as e:
            upload_status.value = f'<span style="color: {COLORS["error"]};">✗ Error reading file: {e}</span>'

upload_btn.observe(on_upload, names='value')


def on_parse(b):
    """Parse DOIs from text input."""
    text = doi_text.value
    if text.strip():
        dois = parse_dois_from_text(text)
        if dois:
            state['dois'] = dois
            status_html.value = f'<span style="color: {COLORS["success"]};">✓ Parsed {len(dois)} valid DOIs</span>'
            process_btn.disabled = False
        else:
            status_html.value = f'<span style="color: {COLORS["error"]};">✗ No valid DOIs found</span>'
    else:
        status_html.value = f'<span style="color: {COLORS["warning"]};">⚠ Please enter DOIs first</span>'

parse_btn.on_click(on_parse)


def on_process(b):
    """Process all DOIs and collect citations."""
    with output:
        clear_output()

        if not state['dois']:
            print("No DOIs to process")
            return

        dois = state['dois']
        api_key = api_key_input.value.strip() or None
        skip_existing = skip_existing_check.value

        print(f"Processing {len(dois)} DOIs...")
        print(f"Skip existing citations: {skip_existing}")
        print("="*50)

        progress.max = len(dois)
        progress.value = 0

        all_results = []
        all_citations = []

        for i, doi in enumerate(dois):
            print(f"\n[{i+1}/{len(dois)}] Processing: {doi}")

            result = process_article_citations(
                doi,
                api_key=api_key,
                skip_existing=skip_existing,
                output_widget=output
            )
            all_results.append(result)
            all_citations.extend(result.get('new_citations', []))

            progress.value = i + 1

        state['results'] = all_results
        state['all_citations'] = all_citations

        # Generate QuickStatements
        if all_citations:
            state['quickstatements'] = generate_batch_quickstatements(all_citations)
            export_btn.disabled = False

        # Print summary
        print("\n" + "="*50)
        print("SUMMARY")
        print("="*50)

        success = sum(1 for r in all_results if r['status'] == 'success')
        not_in_wd = sum(1 for r in all_results if r['status'] == 'not_in_wikidata')
        not_in_s2 = sum(1 for r in all_results if r['status'] == 'not_in_s2')

        print(f"Articles processed: {len(dois)}")
        print(f"  - Successfully processed: {success}")
        print(f"  - Not in Wikidata: {not_in_wd}")
        print(f"  - Not in Semantic Scholar: {not_in_s2}")
        print(f"\nTotal new citations found: {len(all_citations)}")
        print(f"QuickStatements generated: {len(state['quickstatements'])}")

process_btn.on_click(on_process)


def on_export(b):
    """Export QuickStatements to file."""
    if not state['quickstatements']:
        return

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'citation_quickstatements_{timestamp}.txt'

    content = '\n'.join(state['quickstatements'])

    with open(filename, 'w') as f:
        f.write(content)

    # Download in Colab
    try:
        from google.colab import files
        files.download(filename)
    except:
        with output:
            print(f"\n✓ Saved to: {filename}")

export_btn.on_click(on_export)


# Create tabs for input methods
tab_manual = widgets.VBox([
    widgets.HTML('<p>Enter DOIs below, one per line:</p>'),
    doi_text,
    parse_btn
])

tab_upload = widgets.VBox([
    widgets.HTML('<p>Upload a CrossRef CSV export with a DOI column:</p>'),
    upload_btn,
    upload_status
])

input_tabs = widgets.Tab(children=[tab_manual, tab_upload])
input_tabs.set_title(0, 'Manual Entry')
input_tabs.set_title(1, 'Upload CSV')

# Display interface
display(header_html)
display(widgets.HTML('<h4>Step 1: Input DOIs</h4>'))
display(input_tabs)
display(widgets.HTML('<h4>Step 2: Configuration</h4>'))
display(api_key_input)
display(skip_existing_check)
display(status_html)
display(widgets.HTML('<h4>Step 3: Process</h4>'))
display(widgets.HBox([process_btn, export_btn]))
display(progress)
display(output)

## Results Summary

*View detailed results and statistics.*

In [ ]:
def display_results_summary():
    """Display a detailed summary of processing results."""
    if not state['results']:
        print("No results to display. Run processing first.")
        return

    results = state['results']

    print("="*70)
    print("DETAILED RESULTS")
    print("="*70)

    for r in results:
        doi = r['doi']
        status = r['status']

        if status == 'not_in_wikidata':
            print(f"\n✗ {doi}")
            print(f"  Status: Not found in Wikidata")
        elif status == 'not_in_s2':
            source = r['source_article']
            print(f"\n◯ {doi}")
            print(f"  Wikidata: {source['qid']} - {source['label'][:40]}...")
            print(f"  Status: Not found in Semantic Scholar")
        elif status in ['no_references', 'no_dois']:
            source = r['source_article']
            print(f"\n◯ {doi}")
            print(f"  Wikidata: {source['qid']}")
            print(f"  Total references: {r['total_references']}")
            print(f"  References with DOIs: {r['references_with_doi']}")
            print(f"  Status: No citable references found")
        else:
            source = r['source_article']
            print(f"\n✓ {doi}")
            print(f"  Wikidata: {source['qid']} - {source['label'][:40]}...")
            print(f"  S2 Paper ID: {r['s2_paper']['paper_id']}")
            print(f"  Total references: {r['total_references']}")
            print(f"  With DOIs: {r['references_with_doi']}")
            print(f"  In Wikidata: {r['references_in_wikidata']}")
            print(f"  Skipped existing: {r['skipped_existing']}")
            print(f"  New citations: {len(r['new_citations'])}")

    # Statistics
    print("\n" + "="*70)
    print("AGGREGATE STATISTICS")
    print("="*70)

    total_refs = sum(r['total_references'] for r in results)
    total_with_doi = sum(r['references_with_doi'] for r in results)
    total_in_wd = sum(r['references_in_wikidata'] for r in results)
    total_new = len(state['all_citations'])

    print(f"Total references found: {total_refs}")
    print(f"References with DOIs: {total_with_doi} ({100*total_with_doi/max(total_refs,1):.1f}%)")
    print(f"References in Wikidata: {total_in_wd} ({100*total_in_wd/max(total_with_doi,1):.1f}% of DOIs)")
    print(f"New citations to add: {total_new}")


# Run to see results
display_results_summary()

## View Generated QuickStatements

*Preview the QuickStatements before export.*

In [ ]:
def preview_quickstatements(limit=20):
    """Preview generated QuickStatements."""
    qs = state['quickstatements']

    if not qs:
        print("No QuickStatements generated yet.")
        return

    print(f"Generated {len(qs)} QuickStatements")
    print(f"\nShowing first {min(limit, len(qs))}:")
    print("="*70)

    for stmt in qs[:limit]:
        print(stmt)

    if len(qs) > limit:
        print(f"\n... and {len(qs) - limit} more")


preview_quickstatements()

## Export Citations Data as CSV

*Export detailed citation data for analysis.*

In [ ]:
def export_citations_csv():
    """Export citation data as CSV for further analysis."""
    if not state['all_citations']:
        print("No citations to export.")
        return

    df = pd.DataFrame(state['all_citations'])

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'citations_data_{timestamp}.csv'

    df.to_csv(filename, index=False)
    print(f"✓ Exported {len(df)} citations to {filename}")

    # Download in Colab
    try:
        from google.colab import files
        files.download(filename)
    except:
        pass

    return df


# Run to export
# export_citations_csv()

## Batch Processing for Large Sets

*Alternative processing approach for large DOI lists with resume capability.*

In [ ]:
def batch_process_with_checkpoints(dois, checkpoint_file='citation_checkpoint.json', batch_size=50):
    """
    Process DOIs in batches with checkpoint saving for resumability.
    Useful for large sets where you might need to stop and restart.
    """
    # Load existing checkpoint if it exists
    processed = set()
    all_citations = []

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
            processed = set(checkpoint.get('processed', []))
            all_citations = checkpoint.get('citations', [])
        print(f"Loaded checkpoint: {len(processed)} DOIs already processed")

    # Filter to unprocessed DOIs
    remaining = [d for d in dois if d not in processed]
    print(f"Remaining to process: {len(remaining)}")

    # Process in batches
    for i in range(0, len(remaining), batch_size):
        batch = remaining[i:i+batch_size]
        print(f"\nProcessing batch {i//batch_size + 1} ({len(batch)} DOIs)...")

        for doi in batch:
            print(f"  Processing: {doi[:50]}...")

            result = process_article_citations(
                doi,
                skip_existing=True
            )

            if result['status'] == 'success':
                all_citations.extend(result['new_citations'])
                print(f"    ✓ Found {len(result['new_citations'])} new citations")
            else:
                print(f"    - Status: {result['status']}")

            processed.add(doi)

        # Save checkpoint after each batch
        with open(checkpoint_file, 'w') as f:
            json.dump({
                'processed': list(processed),
                'citations': all_citations
            }, f)
        print(f"Checkpoint saved: {len(processed)} DOIs processed")

    # Generate final QuickStatements
    qs = generate_batch_quickstatements(all_citations)

    print(f"\n{'='*50}")
    print(f"BATCH PROCESSING COMPLETE")
    print(f"{'='*50}")
    print(f"Total DOIs processed: {len(processed)}")
    print(f"Total citations found: {len(all_citations)}")
    print(f"QuickStatements generated: {len(qs)}")

    return qs


# Example usage:
# qs = batch_process_with_checkpoints(state['dois'])

## Manual QuickStatements Export

*Manually save the QuickStatements file.*

In [ ]:
def save_quickstatements(filename=None):
    """Save QuickStatements to file."""
    if not state['quickstatements']:
        print("No QuickStatements to save.")
        return

    if not filename:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'citation_quickstatements_{timestamp}.txt'

    content = '\n'.join(state['quickstatements'])

    with open(filename, 'w') as f:
        f.write(content)

    print(f"✓ Saved {len(state['quickstatements'])} statements to {filename}")

    # Try Colab download
    try:
        from google.colab import files
        files.download(filename)
    except:
        pass


# Run to save
# save_quickstatements()